In [3]:
!curl -O https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 1089k  100 1089k    0     0   713k      0  0:00:01  0:00:01 --:--:--  714k


In [6]:
# read it in to inspect it
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [10]:
print("length of dataset in characters: ", len(text))

length of dataset in characters:  1115394


In [12]:
# let's look at the first 1000 characters
print(text[:100])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [13]:
# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [25]:
# create a mapping from characters to integers
stoi = {s:i for i, s in enumerate(chars)}
itos = {i:s for s, i in stoi.items()}

In [38]:
encode = lambda s: [stoi[c] for c in s]
decode = lambda d: ''.join([itos[c] for c in d])

In [41]:
encode("omar")

[53, 51, 39, 56]

In [42]:
decode([53, 51, 39, 56])

'omar'

In [43]:
import torch

data = torch.tensor(encode(text), dtype=torch.long)

In [45]:
encode = lambda s: [stoi[c] for c in s]

torch.Size([1115394])

<b>Makes sense to equal the number of characters in the text, because our tokenizer is at a character level, however
if it were bigger like sub-word tokenizer, it would've been less in the number of tokens, but bigger vocab size. <b>It is a trade off

In [47]:
# train, val split.
n = int(len(text)*0.9)
train_data = data[:n]
val_data = data[n:]
len(train_data), len(val_data)

(1003854, 111540)

<b>The val data is to evaluate our model,to see if the model had overfit the data. 
keep in mind the test set is in the train data. 

It is hard to feed the transformer the entire corpus, that is computationally expensive, we feed it random parts of the text, and it is a fixed size at training and inference, it is called block size. 

In [58]:
block_size = 8 # here is an example
train = train_data[:block_size]
test = train_data[1:block_size+1] 

In [59]:
train, test  

(tensor([18, 47, 56, 57, 58,  1, 15, 47]),
 tensor([47, 56, 57, 58,  1, 15, 47, 58]))

The model is supposed to predict each new character

In [ ]:
for i in range(block_size): 
    x = train[:i+1] # input
    y = test[i] # that is why test is shifted window by size one over the train
    print(f"When input is {x} --> {y}") # input --> output
    
for i in range(block_size):  # same code but displayed as text
    x = train[:i+1] 
    y = test[i] 
    print(f"When input is {decode(x.tolist())} --> {decode([y.item()])}")

<b> The model can handle from one token at at time up to block_size at a time. And the output is fixed one character at a time. 

``` The model can handle from input one up to block size. So when we hit the block size, let's say 
we are predicting over long text then it start to do shift of window by one. 
like: 

     18, 47, 56, 57, 58,  1, 15, 47, suppose next is 65
     47, 56, 57, 58,  1, 15, 47, 65 --> 89
     56, 57, 58,  1, 15, 47, 65, 89 etc

we leave the earliest token and append the new one to maintain a block size, because the model can't be fed more than block size of text, and can output at most one character at a time. 
```

In [142]:
torch.manual_seed(1337)
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----')

for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
----
when input is [24] the target: 43
when input is [24, 43] the target: 58
when input is [24, 43, 58] the target: 5
when input is [24, 43, 58, 5] the target: 57
when input is [24, 43, 58, 5, 57] the target: 1
when input is [24, 43, 58, 5, 57, 1] the target: 46
when input is [24, 43, 58, 5, 57, 1, 46] the target: 43
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target: 39
when input is [44] the target: 53
when input is [44, 53] the target: 56
when input is [44, 53, 56] the target: 1
when input is [44, 53, 56, 1] the target: 58
when input is [44, 53, 56, 1, 58] the target: 46
when input is [44, 53

<b> Now the input is done in batches, since each example is independent, we don't need to train one by one, we can 
    be more efficient by training 4 at a time, and accumulate gradient per batch. However on inference we generally 
    feed one example at a time, unless we are serving bigger number of independent calls, we can feed them into the model in batches. 

In [172]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        
        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T,C)
        
        if targets is None: 
            loss = None
        
        else: 
            B, T, C = logits.shape
        
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            
            loss = F.cross_entropy(logits, targets)

        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx



m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)

print(logits.shape)
print(loss)

torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)


In [240]:
decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist())

'\nJMT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp\nwnYWmnxKWWev-tDqXErVKLgJt-w'

<b> untrained model, so the output is garbage

In [248]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [252]:
batch_size = 32
for steps in range(10000): # increase number of steps for good results...

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())


2.425729990005493


In [256]:
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=500)[0].tolist()))



ING d, matoruee hmaso inge myonisstonen, be,
The we cy.
Anomed weron
Fiurwataltyo tororent have t,'ld tyou to ard an; he: gesily he t-hravo,
Ang, heancour inowisseeay.

ARoube Mond y lorge.
WI mee KERin theicrche igserd't or!
Whas bld piris we r n foo d n ho he corer ur, Be: cther s: at, I RTISTEOMyp ming thed wastt shabrwowhee
Mayowirsafod polld ORI insthands h hir yor'digr t mo hark!
Horond, neadit! orowie th s he istos
lof avoy se RI'd TE m rur ak-
Whis wit s peco br I nshwomavir
BAs ifabear


<b> Somewhat better, bigram model is so weak, because it predicts one character given 
    one input characters, and different character have different output, so if it can only see one character in the past, it lacks context, while it could have general idea, about 
    which characters generally follow each other, each character have like 65 choices, and each one get's chosen depening on context, so if we lack context, we are just predicting on what mostly comes next, which will sound english like but will not make any sense. 

In [258]:
torch.manual_seed(1337)
B, T, C = 4, 8, 2
x = torch.randn(B, T, C)
x.shape

torch.Size([4, 8, 2])

In [313]:
xbow = torch.zeros(B, T, C) # bag of words
for b in range(B): 
    for t in range(T): 
        xprev = x[b, :t+1].mean(0)
        xbow[b, t] = xprev

In [333]:
torch.manual_seed(42)
a = torch.ones(3, 3)
b = torch.randint(0, 10, (3, 2)).float()
c = a @ b
print("a=")
print(a)
print("--")
print("b=")
print(b)
print("--")
print("a=")
print(c)

a=
tensor([[1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 1.]])
--
b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
--
a=
tensor([[14., 16.],
        [14., 16.],
        [14., 16.]])


In [339]:
torch.manual_seed(42)
a = torch.tril(torch.ones(3, 3))
b = torch.randint(0, 10, (3, 2)).float()
c = a @ b
print("a=")
print(a)
print("--")
print("b=")
print(b)
print("--")
print("a=")
print(c)

a=
tensor([[1., 0., 0.],
        [1., 1., 0.],
        [1., 1., 1.]])
--
b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
--
a=
tensor([[ 2.,  7.],
        [ 8., 11.],
        [14., 16.]])


In [346]:
print(a.sum(1, keepdim=True).shape)
print(a.sum(1, keepdim=True))

torch.Size([3, 1])
tensor([[1.],
        [2.],
        [3.]])


In [356]:
torch.tril(torch.ones(8, 8))/torch.tril(torch.ones(8, 8)).sum(1, keepdim=True)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])

In [351]:
#8x 8 @ 8 x 2 = 8x2 

tensor([[[ 0.1808, -0.0700],
         [-0.3596, -0.9152],
         [ 0.6258,  0.0255],
         [ 0.9545,  0.0643],
         [ 0.3612,  1.1679],
         [-1.3499, -0.5102],
         [ 0.2360, -0.2398],
         [-0.9211,  1.5433]],

        [[ 1.3488, -0.1396],
         [ 0.2858,  0.9651],
         [-2.0371,  0.4931],
         [ 1.4870,  0.5910],
         [ 0.1260, -1.5627],
         [-1.1601, -0.3348],
         [ 0.4478, -0.8016],
         [ 1.5236,  2.5086]],

        [[-0.6631, -0.2513],
         [ 1.0101,  0.1215],
         [ 0.1584,  1.1340],
         [-1.1539, -0.2984],
         [-0.5075, -0.9239],
         [ 0.5467, -1.4948],
         [-1.2057,  0.5718],
         [-0.5974, -0.6937]],

        [[ 1.6455, -0.8030],
         [ 1.3514, -0.2759],
         [-1.5108,  2.1048],
         [ 2.7630, -1.7465],
         [ 1.4516, -1.5103],
         [ 0.8212, -0.2115],
         [ 0.7789,  1.5333],
         [ 1.6097, -0.4032]]])

In [359]:
# version 3: use Softmax
tril = torch.tril(torch.ones(T, T))
tril

tensor([[1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1.]])

In [360]:
wei = torch.zeros((T,T))
wei

tensor([[0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.]])

In [364]:
wei = wei.masked_fill(tril == 0, float('-inf'))


tensor([[0., -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0., 0., 0., 0.]])

In [ ]:
wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x
torch.allclose(xbow, xbow3)

batch = 4, t = block_size, c = vocab_size

In [393]:
x, y = get_batch("train")

In [394]:
x.shape 

torch.Size([32, 8])

B =  32, T = 8

In [374]:
emb = nn.Embedding(65, 10) # vocab_size, n_embd
head = nn.Linear(10, 65) # n_emb * vocab_size

In [395]:
token_emb = emb(x) # B, T, C

In [396]:
token_emb.shape

torch.Size([32, 8, 10])

In [403]:
logits = head(token_emb) # B, (T, C) @  T, Vocabsize = (B, T, Vocab) after matmul

In [407]:
B, T, C = logits.shape
logits.shape

torch.Size([32, 8, 65])

In [408]:
logits = logits.view(B*T, C)

In [410]:
logits.shape

torch.Size([256, 65])

In [417]:
y = y.view(B*T)
y[:10]

tensor([15, 53, 56, 47, 53, 50, 39, 52, 42, 59])

In [413]:
F.cross_entropy(logits, y)

tensor(4.2759, grad_fn=<NllLossBackward0>)

In [421]:
pos_emb = nn.Embedding(block_size, 10) #embedding for position
pos_emb

Embedding(8, 10)

In [424]:
x, y = get_batch('train')
B, T = x.shape
B, T

(32, 8)

In [434]:
tok_emb = emb(x)
tok_emb.shape

torch.Size([32, 8, 10])

In [435]:
pos = pos_emb(torch.arange(T))

In [436]:
# 32, 8, 10
# ,   8, 10 broadcast just fine 

In [438]:
tok_pos = tok_emb + pos
tok_pos.shape

torch.Size([32, 8, 10])

In [446]:
# version 4: self-attention!
torch.manual_seed(1337)
B,T,C = 4,8,32 # batch, time, channels
x = torch.randn(B,T,C)

# let's see a single Head perform self-attention
head_size = 16
key = nn.Linear(C, head_size, bias=False) # T, H
query = nn.Linear(C, head_size, bias=False) # T, H
value = nn.Linear(C, head_size, bias=False) # T, H
k = key(x)   # (B, T, H)
q = query(x) # (B, T, H)
wei =  q @ k.transpose(-2, -1) # (B, T, H) @ (B, H, T) ---> (B, T, T)

tril = torch.tril(torch.ones(T, T))
#wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

v = value(x)
out = wei @ v # B, T, T @ (T, H) --> B, T, H
# out = wei @ x

out.shape

torch.Size([4, 8, 16])

In [456]:
print(wei.shape), 
print(v.shape)
print(out.shape)

torch.Size([4, 8, 8])
torch.Size([4, 8, 16])
torch.Size([4, 8, 16])


In [463]:
out[3][7]

tensor([-0.3593,  0.0339, -0.0299, -0.1502, -0.0060, -0.0657, -0.3966, -0.0604,
        -0.5755, -0.2916,  0.1490, -0.0750,  0.0732, -0.0474, -0.6439,  0.2856],
       grad_fn=<SelectBackward0>)